In [39]:
import sys
sys.path.append('..')

import os
import time
import asyncio
import aiohttp
from dotenv import load_dotenv

from weather_api import get_current_temperature, get_current_season, check_temperature_anomaly
from analysis import load_data, get_city_data, compute_seasonal_stats

load_dotenv('../.env')
API_KEY = os.getenv('OPEN_WEATHER_API_KEY')

In [40]:
CITIES = ["Moscow", "London", "Paris", "Tokyo", "New York",
          "Berlin", "Beijing", "Cairo", "Dubai", "Sydney"]

In [41]:
# Синхронно
start = time.time()
results_sync = [get_current_temperature(city, API_KEY) for city in CITIES]
time_sync = time.time() - start

print(f"Синхронно: {time_sync:.3f} сек")
print(results_sync[0])

Синхронно: 3.246 сек
{'city': 'Moscow', 'temperature': 2.59, 'description': 'broken clouds'}


In [42]:
# Асинхронно
async def fetch_all_async(cities, api_key):
    connector = aiohttp.TCPConnector(ssl=False)
    async with aiohttp.ClientSession(connector=connector) as session:
        tasks = [get_current_temperature_async(session, city, api_key) for city in cities]
        return await asyncio.gather(*tasks)

start = time.time()
results_async = await fetch_all_async(CITIES, API_KEY)
time_async = time.time() - start

print(f"Асинхронно: {time_async:.3f} сек")
print(results_async[0])

Асинхронно: 0.302 сек
{'city': 'Moscow', 'temperature': 2.59, 'description': 'broken clouds'}


In [ ]:
print("РЕЗУЛЬТАТЫ ЭКСПЕРИМЕНТА")
print("=" * 50)
print(f"Синхронно: {time_sync:.3f} сек")
print(f"Асинхронно: {time_async:.3f} сек")
print("")
print(f"Ускорение: {time_sync / time_async:.1f}x")

РЕЗУЛЬТАТЫ ЭКСПЕРИМЕНТА
Синхронно:   3.246 сек
Асинхронно:  0.302 сек

Ускорение:   10.7x


In [44]:
# Проверка аномальности текущей температуры


from weather_api import get_current_temperature, get_current_season, check_temperature_anomaly
from analysis import load_data, get_city_data, compute_seasonal_stats

df = load_data('../data/temperature_data.csv')
current_season = get_current_season()
print(f"Текущий сезон: {current_season}\n")


for city in CITIES:
    weather = get_current_temperature(city, API_KEY)
    if "error" in weather:
        print(f"{city}: ошибка - {weather['error']}")
        continue
    
    current_temp = weather["temperature"]
    
    city_df = get_city_data(df, city)
    seasonal_stats = compute_seasonal_stats(city_df)
    season_row = seasonal_stats[seasonal_stats["season"] == current_season].iloc[0]
    
    result = check_temperature_anomaly(
        current_temp, 
        season_row["season_mean"], 
        season_row["season_std"]
    )
    
    status = "аномалия" if result["is_anomaly"] else "норма"
    print(f"{city}: {current_temp:.1f}°C [{status}]")
    print(f"Норма для {current_season}: {result['lower_bound']:.1f}°C  {result['upper_bound']:.1f}°C")
    print()

Текущий сезон: spring

Moscow: 2.6°C [норма]
Норма для spring: -5.0°C  14.7°C

London: 9.3°C [норма]
Норма для spring: 0.9°C  21.4°C

Paris: 6.8°C [норма]
Норма для spring: 2.0°C  22.1°C

Tokyo: 17.2°C [норма]
Норма для spring: 4.7°C  24.9°C

New York: 1.4°C [норма]
Норма для spring: 0.0°C  20.1°C

Berlin: 6.4°C [норма]
Норма для spring: 0.3°C  20.1°C

Beijing: 17.9°C [норма]
Норма для spring: 3.1°C  23.3°C

Cairo: 15.4°C [норма]
Норма для spring: 14.8°C  35.1°C

Dubai: 23.7°C [норма]
Норма для spring: 19.9°C  40.4°C

Sydney: 25.6°C [норма]
Норма для spring: 8.3°C  28.0°C



## Выводы: синхронные vs асинхронные запросы

**Результаты:**
| Метод | Время (10 городов) | Ускорение |
|-------|-------------------|----------|
| Синхронно (requests) | 2.846-3.246 сек | — |
| Асинхронно (aiohttp) | 0.28-0.302 сек | 10-10.7x |

=> асинхронность гораздо быстрее

1. **I/O задача** — большую часть времени ждём ответа от сервера
2. **Параллельные запросы** — asyncio отправляет все запросы одновременно
